# mBERT

This notebook trains a **two-stage hierarchical mBERT model** for multilingual (Bengali + Nepali) hate speech detection.
- **Stage 1 (Binary):** Hate vs. No Hate  
- **Stage 2 (Multi-label):** Insult / Violence / Sexual / Racism / Religious

In [1]:
TEXT_COL     = "Comment"
BINARY_LABEL = "Hate/NoHate"
SUBTYPE_COLS = ["Insult", "Violence", "Sexual", "Racism", "Religious"]
LABEL_COLS   = [BINARY_LABEL] + SUBTYPE_COLS

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128   # mBERT max, we had used 300 in TF-IDF for SVM but we will stick with 128 for BERT
SEED       = 42
OUTPUT_DIR = "mbert_hierarchical_hatespeech"

print("Config loaded.")
print("Label columns:", LABEL_COLS)

Config loaded.
Label columns: ['Hate/NoHate', 'Insult', 'Violence', 'Sexual', 'Racism', 'Religious']


In [ ]:
import importlib, json, os, random, re, subprocess, sys, unicodedata
from pathlib import Path

def ensure_packages(packages):
    missing = []
    for import_name, package_name in packages:
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing.append(package_name)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

ensure_packages([
    ("pandas",      "pandas"),
    ("numpy",       "numpy"),
    ("sklearn",     "scikit-learn"),
    ("torch",       "torch"),
    ("datasets",    "datasets"),
    ("transformers","transformers"),
    ("evaluate",    "evaluate"),
    ("iterstrat",   "iterative-stratification"),
    ("openpyxl",    "openpyxl"),
    ("matplotlib",  "matplotlib"),
    ("seaborn",     "seaborn"),
])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import Dataset as HfDataset, DatasetDict
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import (
    classification_report, f1_score, hamming_loss,
    jaccard_score, multilabel_confusion_matrix,
    precision_recall_fscore_support, roc_curve
)
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, EarlyStoppingCallback,
    Trainer, TrainingArguments,
)

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", DEVICE)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [ ]:
#1 
# Load dataset

DATA_PATH = "../Data/HateSpeechData.xlsx"
df = pd.read_excel(DATA_PATH)

print(f"Using dataset: {DATA_PATH}")
print("Raw shape:", df.shape)
df.head()
print("\nColumns:", list(df.columns))

In [ ]:
#2
#  Standardize column names (strip whitespace)
df.columns = df.columns.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

missing_cols = [c for c in [TEXT_COL] + LABEL_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}. Found: {list(df.columns)}")

# Ensure labels are integer 0/1
for c in LABEL_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Ensure text is string
df[TEXT_COL] = df[TEXT_COL].astype(str)

print("Columns OK:", [TEXT_COL] + LABEL_COLS)
print("Null text rows:", df[TEXT_COL].isna().sum())
print("\nLabel value counts:")
for c in LABEL_COLS:
    print(f"  {c}: {df[c].value_counts().to_dict()}")


In [ ]:
#3
# Fixing inconsistencies and further cleaning the data
#  Rows where a subtype is flagged but binary hate label says No Hate
mask = (df[SUBTYPE_COLS].sum(axis=1) > 0) & (df[BINARY_LABEL] == 0)
print(f"Inconsistent rows found: {mask.sum()} — correcting Hate/NoHate to 1")
df.loc[mask, BINARY_LABEL] = 1

# Verify fix
remaining = ((df[SUBTYPE_COLS].sum(axis=1) > 0) & (df[BINARY_LABEL] == 0)).sum()
print(f"Inconsistent rows remaining after fix: {remaining}")


Here, 

- The SVM used `clean_text()` which **strips all punctuation** with a character whitelist. That's correct for TF-IDF but wrong for mBERT, we know that mBERT was pretrained on natural text that includes `!`, `?`, `"`, `,` etc., and those tokens carry meaning in its embeddings.
- We will, however, **collapse** repeated punctuation (`!!!` → `!`) rather than removing it entirely.
- We do **not** pre-truncate to 300 words — the tokenizer handles truncation at 128 tokens.

In [ ]:
# ── Regex patterns
URL_PATTERN            = re.compile(r"https?://\S+|www\.\S+")
USER_PATTERN           = re.compile(r"@\w+")
HASHTAG_PATTERN        = re.compile(r"#(\w+)")
CONTROL_SPACE_PATTERN  = re.compile(r"[\r\n\t]+")
MULTISPACE_PATTERN     = re.compile(r"\s+")
REPEATED_PUNCT_PATTERN = re.compile(r"([!?.,])\1+")
NEPALI_PUNCT_PATTERN   = re.compile(r"[।॥|]+")
ELLIPSIS_PATTERN       = re.compile(r"\.{2,}")
CHAR_REPEAT_PATTERN    = re.compile(r"(.)\1{2,}")

# Extended emoji block (matches SVM's more thorough coverage)
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\U00002600-\U000026FF"
    "\U00002700-\U000027BF"
    "]+", flags=re.UNICODE
)

def normalize_text(text: str) -> str:
    """
    DL-safe cleaning for mBERT. Key differences from SVM clean_text():
      - Does NOT strip punctuation (mBERT needs it)
      - Does NOT pre-truncate (tokenizer handles MAX_LENGTH=128)
      - DOES remove emojis, normalize Unicode, replace social tokens
    """
    text = unicodedata.normalize("NFKC", str(text))

    # Remove zero-width / BOM characters
    text = text.replace("\u200b", " ").replace("\ufeff", " ")

    # Collapse control whitespace
    text = CONTROL_SPACE_PATTERN.sub(" ", text)

    # Replace social tokens with placeholders mBERT understands
    text = URL_PATTERN.sub(" [URL] ", text)
    text = USER_PATTERN.sub(" [USER] ", text)
    text = HASHTAG_PATTERN.sub(r" \1 ", text)   # unwrap hashtag, keep word

    # Remove emojis (they tokenize to [UNK] in mBERT — noise, not signal)
    text = EMOJI_PATTERN.sub(" ", text)

    # Remove Nepali/Bengali sentence-end punctuation (not meaningful to mBERT)
    text = NEPALI_PUNCT_PATTERN.sub(" ", text)

    # Remove ellipsis ("...") — not a meaningful mBERT token
    text = ELLIPSIS_PATTERN.sub(" ", text)

    # Collapse spam repetition ("hahahaha" → "ha", "!!!" → "!")
    text = CHAR_REPEAT_PATTERN.sub(r"\1", text)
    text = REPEATED_PUNCT_PATTERN.sub(r"\1", text)

    # Final whitespace collapse
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

# Apply to dataframe
before_rows = len(df)
df["text"] = df[TEXT_COL].apply(normalize_text)

# Drop rows that became empty after cleaning
df = df[df["text"].str.strip().str.len() > 0].copy()

# Drop rows with fewer than 2 tokens (not useful for any model)
df = df[df["text"].str.split().str.len() >= 2].copy()

# Drop duplicates (same cleaned text + same label combination)
df = df.drop_duplicates(subset=["text"] + LABEL_COLS).reset_index(drop=True)

print(f"Rows before cleaning: {before_rows}")
print(f"Rows after cleaning:  {len(df)}  (removed {before_rows - len(df)})")
print(f"\nSample before → after:")
display(df[[TEXT_COL, "text"]].head(8))

In [ ]:
#4
# Here, we will do a quick sanity check on cleaned dataset before we train-test split 

df["char_len"]        = df["text"].str.len()
df["word_count"]      = df["text"].str.split().str.len()
df["label_cardinality"] = df[LABEL_COLS].sum(axis=1)

print("=== Dataset shape:", df.shape)
print("\n--- Binary label distribution ---")
counts = df[BINARY_LABEL].value_counts().sort_index()
for k, v in counts.items():
    print(f"  {'Hate' if k==1 else 'No Hate'}: {v} ({v/len(df)*100:.1f}%)")

print("\n--- Subtype label prevalence (% of total rows) ---")
for c in SUBTYPE_COLS:
    print(f"  {c}: {df[c].mean()*100:.2f}%")

print("\n--- Text length stats (cleaned) ---")
display(df[["char_len", "word_count"]].describe().round(2))

# Label distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.plot(kind="bar", ax=axes[0], color=["steelblue", "tomato"])
axes[0].set_title("Hate vs No Hate")
axes[0].set_xticklabels(["No Hate", "Hate"], rotation=0)
axes[0].set_ylabel("Count")

df[SUBTYPE_COLS].sum().sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Subtype Label Counts")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


We use `MultilabelStratifiedShuffleSplit` (80/10/10) so each label's positive ratio is preserved across all three splits. Standard `train_test_split` does not guarantee this for multi-label data.

In [ ]:
#5
# Split Data
labels_matrix = df[LABEL_COLS].values

outer_splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(outer_splitter.split(df["text"], labels_matrix))

train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df  = df.iloc[temp_idx].reset_index(drop=True)

inner_splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel_idx, test_rel_idx = next(inner_splitter.split(temp_df["text"], temp_df[LABEL_COLS].values))

val_df  = temp_df.iloc[val_rel_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_rel_idx].reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows":  [len(train_df), len(val_df), len(test_df)],
})
split_summary["ratio"] = (split_summary["rows"] / len(df)).round(4)
display(split_summary)

print("\nLabel prevalence per split (should be similar across all three):")
for name, sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name}: {sdf[LABEL_COLS].mean().round(3).to_dict()}")


In [ ]:
#6
# Loading mBERT Tokenizer and Demo 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quick demo: see how mBERT tokenizes your actual languages
demo_texts = [
    "यो मान्छे धेरै नराम्रो छ!",   # Nepali
    "এই লোকটা খুব খারাপ।",          # Bengali
    "You are an idiot.",             # English
]
for text in demo_texts:
    enc = tokenizer(text, truncation=True, max_length=32, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    print(f"Text   : {text}")
    print(f"Tokens : {tokens}")
    print("-" * 60)

In [ ]:
# 7
# Our dataset is imbalanced (more No Hate than Hate, and rare subtypes like Sexual/Racism). 
# We will compute weights from the training split only to avoid any data leakage.


# Binary class weights (for CrossEntropyLoss in Stage 1)
binary_counts = train_df[BINARY_LABEL].value_counts().reindex([0, 1], fill_value=0)
binary_class_weights = len(train_df) / (2.0 * np.maximum(binary_counts.values.astype(np.float32), 1.0))

# Subtype positive weights (for BCEWithLogitsLoss in Stage 2)
subtype_train_df = train_df[train_df[BINARY_LABEL] == 1].reset_index(drop=True)
subtype_val_df   = val_df[val_df[BINARY_LABEL] == 1].reset_index(drop=True)
subtype_test_df  = test_df[test_df[BINARY_LABEL] == 1].reset_index(drop=True)

if min(len(subtype_train_df), len(subtype_val_df), len(subtype_test_df)) == 0:
    raise ValueError("A subtype split is empty — increase data or adjust split ratios.")

subtype_positive  = subtype_train_df[SUBTYPE_COLS].sum(axis=0).values.astype(np.float32)
subtype_negative  = (len(subtype_train_df) - subtype_positive).astype(np.float32)
subtype_pos_weight = np.clip(subtype_negative / np.maximum(subtype_positive, 1.0), 1.0, 25.0)

display(pd.DataFrame({
    "class": ["NoHate", "Hate"],
    "count":        binary_counts.values.astype(int),
    "class_weight": binary_class_weights.round(3),
}))

display(pd.DataFrame({
    "label":             SUBTYPE_COLS,
    "positive_examples": subtype_positive.astype(int),
    "pos_weight":        subtype_pos_weight.round(3),
}))


In [ ]:
def frame_to_binary_dataset(frame):
    return HfDataset.from_dict({
        "text":   frame["text"].tolist(),
        "labels": frame[BINARY_LABEL].astype(int).tolist(),
    })

def frame_to_subtype_dataset(frame):
    frame = frame[frame[BINARY_LABEL] == 1].reset_index(drop=True)
    return HfDataset.from_dict({
        "text":   frame["text"].tolist(),
        "labels": frame[SUBTYPE_COLS].astype(np.float32).values.tolist(),
    })

binary_dataset = DatasetDict({
    "train":      frame_to_binary_dataset(train_df),
    "validation": frame_to_binary_dataset(val_df),
    "test":       frame_to_binary_dataset(test_df),
})

subtype_dataset = DatasetDict({
    "train":      frame_to_subtype_dataset(train_df),
    "validation": frame_to_subtype_dataset(val_df),
    "test":       frame_to_subtype_dataset(test_df),
})

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

binary_tokenized  = binary_dataset.map(tokenize_batch,  batched=True, remove_columns=["text"])
subtype_tokenized = subtype_dataset.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

display(pd.DataFrame({
    "split":        ["train", "validation", "test"],
    "binary_rows":  [len(train_df), len(val_df), len(test_df)],
    "subtype_rows": [len(subtype_train_df), len(subtype_val_df), len(subtype_test_df)],
}))
print("Datasets tokenized and ready.")

In [ ]:
#8
#Defining our Custom Trainers and Metric functions 

# ── Threshold placeholders (tuned in Cell 15 after training)
binary_threshold   = 0.5
subtype_thresholds = np.full(len(SUBTYPE_COLS), 0.5, dtype=np.float32)

# ── Numpy helpers
def softmax_np(logits):
    logits  = np.asarray(logits)
    shifted = logits - logits.max(axis=1, keepdims=True)
    e       = np.exp(shifted)
    return e / e.sum(axis=1, keepdims=True)

def sigmoid_np(logits):
    return 1.0 / (1.0 + np.exp(-np.asarray(logits)))

# ── Metric functions (called by Trainer during eval)
def compute_binary_metrics(eval_pred):
    logits, labels = eval_pred
    labels = labels.astype(int)
    probs  = softmax_np(logits)[:, 1]
    preds  = (probs >= binary_threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"binary_f1": float(f1), "binary_precision": float(p),
            "binary_recall": float(r), "binary_accuracy": float((preds==labels).mean())}

def compute_subtype_metrics(eval_pred):
    logits, labels = eval_pred
    labels = labels.astype(int)
    probs  = sigmoid_np(logits)
    preds  = (probs >= subtype_thresholds).astype(int)
    p_mi, r_mi, f1_mi, _ = precision_recall_fscore_support(labels, preds, average="micro", zero_division=0)
    p_ma, r_ma, f1_ma, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"micro_f1": float(f1_mi), "macro_f1": float(f1_ma),
            "micro_precision": float(p_mi), "micro_recall": float(r_mi)}

# ── Custom Trainer: Stage 1 (binary, weighted cross-entropy)
class BinaryTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = None if class_weights is None else torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels  = inputs.pop("labels").long()
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        )
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Custom Trainer: Stage 2 (multi-label, weighted BCE)
class MultiLabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = None if pos_weight is None else torch.tensor(pos_weight, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels  = inputs.pop("labels").float()
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(outputs.logits.device) if self.pos_weight is not None else None
        )
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

print("Custom trainers and metric functions defined.")

In [ ]:
# 9
# Initializing models and training arguments

binary_id2label  = {0: "NoHate", 1: "Hate"}
binary_label2id  = {v: k for k, v in binary_id2label.items()}
subtype_id2label = {i: l for i, l in enumerate(SUBTYPE_COLS)}
subtype_label2id = {l: i for i, l in subtype_id2label.items()}

binary_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label=binary_id2label, label2id=binary_label2id,
).to(DEVICE)

subtype_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(SUBTYPE_COLS),
    problem_type="multi_label_classification",
    id2label=subtype_id2label, label2id=subtype_label2id,
).to(DEVICE)

BATCH_SIZE      = 8  if torch.cuda.is_available() else 4
EVAL_BATCH_SIZE = 16 if torch.cuda.is_available() else 4
GRAD_ACCUM      = 2  if torch.cuda.is_available() else 1

def build_training_args(output_dir, metric_for_best_model):
    kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.1,
        weight_decay=0.01,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=True,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )
    # Handle API differences across transformers versions
    params = set(TrainingArguments.__init__.__code__.co_varnames)
    kwargs["evaluation_strategy" if "evaluation_strategy" in params else "eval_strategy"] = "epoch"
    if "gradient_checkpointing" in params:
        kwargs["gradient_checkpointing"] = torch.cuda.is_available()
    if "group_by_length" in params:
        kwargs["group_by_length"] = True
    return TrainingArguments(**kwargs)

binary_training_args  = build_training_args(str(Path(OUTPUT_DIR) / "stage1_hate"),    "binary_f1")
subtype_training_args = build_training_args(str(Path(OUTPUT_DIR) / "stage2_subtypes"), "micro_f1")

print("Models and training arguments ready.")
print(f"  Binary model params:  {binary_model.num_parameters():,}")
print(f"  Subtype model params: {subtype_model.num_parameters():,}")